In [ ]:
!pip install sentence-transformers faiss-cpu google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 36.0 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
from google import genai

In [ ]:
from google import genai
from google.colab import userdata


client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))


In [ ]:
docs = [
    "I am anshif.i am from Tiruppur",
    "i am studying in karpagam college of engineering.it ios in othakalmandapam coimbatore",
    "I am currently pursuing Artificial intelligence and data science.Department is in the Old D Block",
    "i am going to get a job for 45 lpa"
]

In [ ]:
model=SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(docs).astype("float32")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
doc_embeddings.shape

(5, 384)

In [ ]:
dimension=doc_embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

In [ ]:
query=["What i am pursuing"]
query_embedding=model.encode(query).astype("float32")

top_k=2
distances,indices=index.search(query_embedding,top_k)

retrieved_chunks=[docs[i] for i in indices[0]]
print("Retrieved Chunks:")

for chunk in retrieved_chunks:
  print("-",chunk)

Retrieved Chunks:
- I am currently pursuing Artificial intelligence and data science.Department is in the Old D Block
- i am studying in karpagam college of engineering.it ios in othakalmandapam coimbatore


In [ ]:
context="\n".join(retrieved_chunks)
prompt=f"""
Answer the question based only on the context below.
Context:
{context}
Question:
{query}
Answer:
"""
response=client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config={"temperature":0.2}
)
print("Final Answer:")
print(response.text)



Final Answer:
Artificial intelligence and data science
